In [1]:
import plotly
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
import re
from scipy.signal import find_peaks

import utils_for_plotly_rewrites
import nist_codes

In [2]:
# Shared Values
Y_TITLE = 'Intensity'
X_TITLE = 'Wavelength (nm)'
COLOUR = 'white'
BG = 'black'
X_MIN = 400
X_MAX = 750
FIG_WIDTH = 15
DPI = 600

# Traditional Plot Values
FIG_HEIGHT_BASE = 3.0
FIG_SIZE = (FIG_WIDTH, FIG_HEIGHT_BASE)
MIN_NEEDLE_WIDTH = 0.1
MAX_NEEDLE_WIDTH = 0.3
MAX_Y_SCALE = 0.75
NEEDLE_POWER_SHAPE = 4
LABEL_NORM_INT = 0.20
GLOW_WIDTH_MULT = 1.3

# Dynamic Height Values (overflow section)
fig_height_overflow_scale = 9.0

# Normalised-Specific Values
NORM_PROM_PERC = 0.15
NORM_MIN_BRIGHT = 0.01
NORM_GLOW_ALPHA = 0
NORM_PEAK_EMPHASIS = 1.1
NORM_PEAK_LABEL_POSN = 0.75

# "Default" Values (i.e. for not normalised)
DEFAULT_PROM_PERC = 0.08    # also used by "other" plots
DEFAULT_MIN_BRIGHT = 0.1    # also used by "other" plots
DEFAULT_GLOW_ALPHA = 0.35
DEFAULT_PEAK_EMPHASIS = 1.4
DEFAULT_PEAK_LABEL_POSN = 0.77

# Other Plot Values
fig_size = (15,6)
#prominence = 0.08       # changed from 0.12
#min_bright = 0.1
min_alpha = 0.1
min_alpha_scatter = 0.2
base_marker_size = 5
max_marker_size_factor = 95
gamma_factor = 0.8
bar_width = 1
smoothing_window = 5    # Increase this value to control the degree of smoothing
base_sigma_nm = 0.5 # Base width of the Gaussian (for low intensity peaks)
max_sigma_multiplier = 4.0 # How much wider the highest intensity peaks can be
reverse_x = True
plot_type = None
show_grid = True

In [3]:
lambda_tokens = ["nm", "wavelength", "wavelength_nm", "lambda", "lambda_nm", "wl", "wl_nm", "Observed", "Observed Wavelength", "obs", "wave", "w"]

int_tokens =["Grey Val", "grey val", "gray val", "grayscale", "gray value", "intensity", "signal", "counts", "value", "int", "rel. int.", "grey", "Rel. Int.", "Relative Intensity", "Rel Int", "Intensity", "A", "Aki", "gA", "gf", "weighted f", "f", "Intensity/Counts", 'rel', 'count', 'flux', 'grey value', 'i']


In [4]:
detection_col = '_raw_int'
int_col = '_adj_int'

In [31]:
def resolve_column(df, candidates, label):
    headers = [(str(col).strip(), str(col).strip().lower()) for col in df.columns]

    # Build a flat keyword list from all candidates.
    # Each candidate may be a phrase; we split on non-alphanumeric characters.
    keywords = []
    for candidate in candidates:
        text = str(candidate).strip().lower()
        if not text:
            continue
        parts = [part for part in re.split(r'[^a-z0-9]+', text) if part]
        keywords.extend(parts if parts else [text])

    # Prefer exact matches first, then keyword containment.
    for original, normalized in headers:
        for candidate in candidates:
            candidate_norm = str(candidate).strip().lower()
            if candidate_norm and normalized == candidate_norm:
                return original
    for original, normalized in headers:
        for keyword in keywords:
            if keyword and keyword in normalized:
                return original
    raise KeyError("Could not find a", label, "column. Available columns:", (df.head()))


def res_col_names(data_df, detect_columns, nm_col, int_col):
    global wl_col
    global INT_col

    def normalize_selected_col(value):
        if value is None:
            return None

        text = str(value).strip()
        if text == "":
            return None

        if text.isdigit():
            index = int(text)
            if 0 <= index < len(data_df.columns):
                return data_df.columns[index]
            return None

        for col in data_df.columns:
            if str(col).strip() == text:
                return col

        for col in data_df.columns:
            if str(col).strip().lower() == text.lower():
                return col
        #return None

    if detect_columns is True:
        wl_col = normalize_selected_col(nm_col)
        INT_col = normalize_selected_col(int_col)
        if wl_col is None or INT_col is None:
            return data_df, None, None, True

        return data_df, wl_col, INT_col, False

    try:
        wl_col = resolve_column(data_df, lambda_tokens, "wavelength")
        INT_col = resolve_column(data_df, int_tokens, "intensity")
        return data_df, wl_col, INT_col, False
    except KeyError:
        return data_df, None, None, True

def compute_label_positions(peak_nms, y_step, intensities=None, base_y=None, min_sep_nm=0.5, method="prefer_stronger_top", max_y=0.98):
    if not peak_nms:
        return []

    # prepare indices sorted by wavelength
    idx_sorted = sorted(range(len(peak_nms)), key=lambda i: peak_nms[i])
    result = [base_y] * len(peak_nms)

    # build clusters of peaks closer than min_sep_nm
    clusters = []
    cur = [idx_sorted[0]]
    for i in idx_sorted[1:]:
        if abs(peak_nms[i] - peak_nms[cur[-1]]) <= min_sep_nm:
            cur.append(i)
        else:
            clusters.append(cur)
            cur = [i]
    clusters.append(cur)


    for cluster in clusters:
        if len(cluster) == 1:
            result[cluster[0]] = base_y
            continue

        if method == "prefer_stronger_top" and intensities is not None:
            cluster_sorted = sorted(cluster, key=lambda k: -float(intensities[k]))
            n = len(cluster_sorted)
            if n == 1:
                result[cluster_sorted[0]] = base_y
            else:
                step = min(y_step, (max_y - base_y) / (n - 1))
                for pos, idx in enumerate(cluster_sorted):
                    result[idx] = base_y + pos * step

    return result

In [6]:
def run_nist_check(data_df, detect_columns, nm_col, int_col, force_nist=None):
    res_col_names(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    df_data = data_df.copy()
    df_data[wl_col] = pd.to_numeric(df_data[wl_col], errors='coerce')
    has_any_nist, _nist_diag = nist_codes.detect_nist_values(df_data[INT_col], force_nist=force_nist)
    return df_data, has_any_nist, detect_columns, nm_col, int_col

def prepare_generic_spectrum(df, int_col, apply_descriptor_adjustments=False):
    df = df.copy()
    df['_raw_int'] = pd.to_numeric(df[int_col], errors='coerce')
    df['_descriptor'] = ''
    df['_intensity_mult'] = 1.0
    df['_width_mult'] = 1.0
    df['_include'] = True
    df['_adj_int'] = df['_raw_int']
    return df


def prep_with_nist(data_df, detect_columns, nm_col, int_col, x_min = utils_for_plotly_rewrites.X_MIN, x_max = utils_for_plotly_rewrites.X_MAX, apply_descriptor_adjustments = False):
    global int_range, Y_MAX

    df_plot_data, has_any_nist, _, _, _ = run_nist_check(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    if has_any_nist:
        df_plot_data = nist_codes.prepare_nist_spectrum(df_plot_data, INT_col, apply_descriptor_adjustments)
        print('NIST Destriptors Detected. Preparing NIST Rendering.')
    else:
        df_plot_data = prepare_generic_spectrum(df_plot_data, INT_col, apply_descriptor_adjustments)
        print('No NIST Destriptors Detected. Preparing Generic Rendering.')

    # parse NIST-style cells
    parsed_intensity = df_plot_data[INT_col].apply(nist_codes.parse_nist_intensity)
    df_plot_data['_raw_int'] = parsed_intensity.apply(lambda t: t[0])
    df_plot_data['_descriptor'] = parsed_intensity.apply(lambda t: t[1])

    # drop rows missing wavelength or numeric intensity
    df_plot_data = df_plot_data.dropna(subset=[wl_col, '_raw_int']).copy()
    df_plot_data = df_plot_data[(df_plot_data[wl_col] >= x_min) & (df_plot_data[wl_col] <= x_max)].copy()
    df_plot_data = df_plot_data.sort_values(by=wl_col).reset_index(drop=True)

    # returns empty flag for empty datasets
    if df_plot_data.empty:
        return None, True, False 

    # Normalize using adjusted intensity
    min_int_val = df_plot_data['_adj_int'].min()
    max_int_val = df_plot_data['_adj_int'].max()
    int_range = max_int_val - min_int_val

    int_vals = df_plot_data['_adj_int']
    Y_MAX = int_vals.max()
    print('(prep_with_nist) ymax = ', Y_MAX)
    
    if int_range == 0 or np.isnan(int_range):
        df_plot_data['Norm_Int'] = 1.0
    else:
        df_plot_data['Norm_Int'] = (df_plot_data['_adj_int'] - min_int_val) / int_range

    return df_plot_data, False, has_any_nist

In [7]:
HE_SHEET = 'he test.xlsx'
NIST_SHEET = 'oxygen nist 2.xlsx'
HE_DF = pd.read_excel(HE_SHEET)
NIST_DF = pd.read_excel(NIST_SHEET)

In [8]:
major_locator = {
    "tickmode": "linear",
    "tick0": 0,
    "dtick": 50
}

minor_locator = {
    "tickmode": "linear",
    "tick0": 0,
    "dtick": 10
}

In [33]:
def trad_spec_labels(
    has_any_nist,
    scale_mode,
    title=None,
    random_title=None,
    x_title=utils_for_plotly_rewrites.X_TITLE,
    x_range=[utils_for_plotly_rewrites.X_MIN, utils_for_plotly_rewrites.X_MAX],
    show_grid=False,
    bg_colour = utils_for_plotly_rewrites.BG,
    text_colour=utils_for_plotly_rewrites.COLOUR,
    grid_color='#333333', # Adjusted to a very dark grey
    dtick_x=50, # From major_locator
    ):

    if has_any_nist or scale_mode == 'raw': 
        pad = 15
    else:
        pad = 10

    if title:
        go.indicator.title(str(title).strip(), color = text_colour, y=0.98, pad=pad)
    if random_title:
        go.indicator.title(str(utils_for_plotly_rewrites.generate_random_title()), color=text_colour, y=0.98, pad=pad)


    layout_config = dict (
        plot_bgcolor='black',
        paper_bgcolor='black',
        width = utils_for_plotly_rewrites.FIG_WIDTH * 80,
        height = utils_for_plotly_rewrites.FIG_HEIGHT_BASE * 80,
        font=dict(
            color=text_colour,
            #family = 'DejaVu Sans, sans-serif',            # can only be set in .js
            ),
        xaxis=dict(
            range=[X_MIN, X_MAX],
            gridcolor=bg_colour,
            showgrid=False,
            title=x_title,
            dtick=dtick_x,
            ticks='outside',
            tickcolor=text_colour,
            minor = dict(dtick=10, ticks='outside', tickcolor=text_colour, tickwidth=0.5),
            showline=False,
        ),
        yaxis=dict(
            range=[0,1],
            visible=False
        )
    )

    return layout_config

    ## ax.tick_params(axis='x', which='major', colors=text_color, labelsize=10)
    ## ax.tick_params(axis='x', which='minor', colors=text_color, length=4, width=0.5)


In [10]:
def plot_trad(
    df_plot_data,
    scale_mode,
    detect_columns,
    nm_col,
    has_any_nist,
    show_peak_labels,
    title,
    random_title,
    show_grid = False,
    scale_by_int = None,
    save_path = None,
    prominence_percentage=0,     
    fig_size=utils_for_plotly_rewrites.FIG_SIZE,
    min_brightness=0,
    peak_wavelengths=None,
    x_min=utils_for_plotly_rewrites.X_MIN,
    x_max=utils_for_plotly_rewrites.X_MAX,
    min_needle_max_width_nm=utils_for_plotly_rewrites.MIN_NEEDLE_WIDTH,
    max_needle_max_width_nm=utils_for_plotly_rewrites.MAX_NEEDLE_WIDTH,
    needle_shape_power=utils_for_plotly_rewrites.NEEDLE_POWER_SHAPE,
    glow_width_multiplier=utils_for_plotly_rewrites.GLOW_WIDTH_MULT,
    glow_alpha=0,
    max_needle_y_scale=0,
    dpi=utils_for_plotly_rewrites.DPI,
    peak_label_y_position=0,
    label_min_norm_int=utils_for_plotly_rewrites.LABEL_NORM_INT,
    subplots_adjust_top = 0.90,
    max_needle_y = utils_for_plotly_rewrites.MAX_Y_SCALE,
    fig_height_overflow_scale = 9.0 * 80,
    fig_height_base = utils_for_plotly_rewrites.FIG_HEIGHT_BASE * 80,
):

    if show_peak_labels is False:
        max_needle_y_scale = 0.98
    else:
        max_needle_y_scale = utils_for_plotly_rewrites.MAX_Y_SCALE

    trad_fig = go.Figure()


    #new_utils.trad_spec_labels(fig=fig, ax=ax, x_min=x_min, x_max=x_max, title=title, random_title=random_title, has_any_nist=has_any_nist, scale_mode=scale_mode)

    if has_any_nist:
        scale_mode = None
        min_brightness = utils_for_plotly_rewrites.DEFAULT_MIN_BRIGHT
        glow_alpha = utils_for_plotly_rewrites.DEFAULT_GLOW_ALPHA
        prominence_percentage = utils_for_plotly_rewrites.DEFAULT_PROM_PERC
        peak_label_y_position = max_needle_y + 0.01
    else:
        if scale_mode == 'raw':
            glow_alpha = utils_for_plotly_rewrites.DEFAULT_GLOW_ALPHA
            peak_label_y_position = utils_for_plotly_rewrites.DEFAULT_PEAK_LABEL_POSN
        elif scale_mode == 'normalize':
            glow_alpha = utils_for_plotly_rewrites.NORM_GLOW_ALPHA
            peak_label_y_position = utils_for_plotly_rewrites.NORM_PEAK_LABEL_POSN

    # --- Peak Detection ---
    if peak_wavelengths is not None:
        nm_vals = df_plot_data[wl_col].values
        peaks = []
        for pw in peak_wavelengths:
            try:
                pv = float(pw)
            except Exception:
                continue
            idx = int(np.argmin(np.abs(nm_vals - pv)))
            peaks.append(idx)
        peaks = sorted(set(peaks))
    elif has_any_nist:
        print ("DEBUG: has nist plot trad", has_any_nist)
        peaks = nist_codes.identify_spectral_peaks(df_plot_data.reset_index(drop=True), prominence_percentage, peak_wavelengths=peak_wavelengths)
    else:
        raw_min = df_plot_data[detection_col].min()
        raw_max = df_plot_data[detection_col].max()
        raw_range = raw_max - raw_min
        if scale_mode == 'raw':
            prominence_percentage = utils_for_plotly_rewrites.DEFAULT_PROM_PERC
        if scale_mode == 'normalize':
            prominence_percentage = utils_for_plotly_rewrites.NORM_PROM_PERC
        dynamic_prominence = prominence_percentage * (raw_range if raw_range != 0 else 1.0)
        peaks, _ = find_peaks(df_plot_data[INT_col], prominence=dynamic_prominence)
    
    peak_nms = [float(df_plot_data.iloc[index][wl_col]) for index in peaks]
    peak_ints = [float(df_plot_data.iloc[index]["Norm_Int"]) for index in peaks]
    init_peak_label_y_posn = peak_label_y_position

    if show_peak_labels is True:
        try:
            label_ys = compute_label_positions(
                peak_nms,
                intensities=peak_ints,
                base_y=init_peak_label_y_posn,
                min_sep_nm=0.5,
                y_step=0.08,
                method="prefer_stronger_top",
                max_y=0.98,
            )
        except Exception:
            label_ys = [init_peak_label_y_posn] * len(peaks)

        if label_ys:
            max_label_y = max(label_ys)
            overflow = max(0, max_label_y - init_peak_label_y_posn)
            
            print("\noverflow:", overflow)
            print("o.g. peak label y pos'n:", init_peak_label_y_posn)
            if overflow > 0:
                new_height = fig_height_base + overflow * fig_height_overflow_scale
                new_fig_height = new_height
                trad_fig.update_layout(height=float(new_fig_height))
                current_needle_height_in = max_needle_y * new_height
                new_needle_height = (max_needle_y * fig_height_base) / new_height 
                max_needle_y_scale = new_needle_height + 0.04
                new_peak_y_posn = (init_peak_label_y_posn * fig_height_base) / new_height
                peak_label_y_position = new_peak_y_posn

                print("\nog max needle y:", max_needle_y, "\nmax label y:", max_label_y, "\ninit peak label y:", init_peak_label_y_posn, "\n\t label ys", label_ys)
                print("needle height goal:", new_needle_height, "\n \t inches:", new_needle_height*new_height)
                print("current needle height (in):", current_needle_height_in)
                print("height:", new_height, "max label y:", max_label_y)
                print("\npeak label y pos'n goal:", new_peak_y_posn, "\nnew fig height:", new_fig_height,)      
            else:
                print("Labels fit — no expansion needed")
                max_needle_y_scale = max_needle_y
                print("peak label y pos'n:", peak_label_y_position)
        else:
            max_needle_y_scale = max_needle_y
            print("No labels on this spectrum")
        print(f"DEBUG: max_label_y = {max(label_ys) if label_ys else 'N/A'}")

        try:
            label_ys = compute_label_positions(
                peak_nms,
                intensities=peak_ints,
                base_y=peak_label_y_position,   # now uses updated value
                min_sep_nm=0.4,
                y_step=0.15,
                method="prefer_stronger_top",
                max_y=0.98,
            )
        except Exception:
            label_ys = [peak_label_y_position] * len(peaks)
        print("\n new peak label pos'n:", peak_label_y_position)

    # Render every transition as a faint needle (increase visibility for verification)
    _bg_y_top = max_needle_y_scale  # use full height for visibility
    _bg_y = np.linspace(0, _bg_y_top, 40)
    for _idx, _row in df_plot_data.iterrows():
        _nm = float(_row[nm_col])
        _ni = float(_row.get('Norm_Int', 0.0))
        _base_rgb = utils_for_plotly_rewrites.rgb(_nm)
        _final_scale = utils_for_plotly_rewrites.final_scale(min_brightness, _ni)
        _color = (_base_rgb[0] * _final_scale, _base_rgb[1] * _final_scale, _base_rgb[2] * _final_scale)
        _width_mult = float(_row.get('_width_mult', 1.0))
        _colour = f"rgb({int(_color[0]*255)}, {int(_color[1]*255)}, {int(_color[2]*255)})"
        _bg_base_width = min_needle_max_width_nm * 1.0 * _width_mult   # make background lines thicker for test
        _bg_widths = _bg_base_width * np.ones_like(_bg_y)
        _x_left = utils_for_plotly_rewrites.left_x(_nm, _bg_widths)
        _x_right = utils_for_plotly_rewrites.right_x(_nm, _bg_widths)
        #ax.fill_betweenx(_bg_y, _x_left, _x_right, facecolor=_color, alpha=glow_alpha, edgecolor='none', linewidth=0, zorder=0)

        loop_x = list(_x_left) + list(_x_right)[::-1]
        loop_y = list(_bg_y) + list(_bg_y)[::-1]

        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=_colour,
            opacity=glow_alpha,
            line=dict(width=0),      # Equivalent to edgecolor='none', linewidth=0
            mode='none',             # Hides individual markers/lines, showing only the fill
            showlegend=False,
            hoverinfo='skip'         # Keeps background elements from triggering tooltips
        ))

    # Plot sharp distinct lines for each identified peak using fill_betweenx for needle shape
    for j, peak_index in enumerate(peaks):
        peak_nm = df_plot_data.iloc[peak_index][nm_col]
        peak_norm_int = df_plot_data.iloc[peak_index]['Norm_Int']
        base_rgb = utils_for_plotly_rewrites.rgb(peak_nm)

        # Emphasize peaks: gentle gamma + emphasis multiplier for labeled peaks
        _peak_gamma = 0.8
        if has_any_nist or scale_mode == 'raw':   
            _peak_emphasis = utils_for_plotly_rewrites.DEFAULT_PEAK_EMPHASIS
            min_brightness = utils_for_plotly_rewrites.DEFAULT_MIN_BRIGHT
        elif scale_mode == 'normalize':
            _peak_emphasis = utils_for_plotly_rewrites.NORM_PEAK_EMPHASIS
            min_brightness = utils_for_plotly_rewrites.NORM_MIN_BRIGHT
        pre_int_scale = min_brightness + (1 - min_brightness) * (peak_norm_int ** _peak_gamma)
        final_int_scale = min(1.0, pre_int_scale * _peak_emphasis)
        color_rgb = utils_for_plotly_rewrites.colored_rgb(base_rgb, final_int_scale)

        # Scale the maximum width of the needle based on normalized intensity
        base_width = min_needle_max_width_nm + (max_needle_max_width_nm - min_needle_max_width_nm) * peak_norm_int
        width_mult = df_plot_data.iloc[peak_index].get('_width_mult', 1.0)

        # boost peak widths so labeled peaks stand out
        _peak_width_boost = 1.6
        scaled_max_width_nm = base_width * width_mult * _peak_width_boost
        
        # Calculate the actual height the needle should reach (fraction of 0-1)
        current_peak_render_height = max_needle_y_scale

        # Define y-coordinates for the needle shape, spanning from 0 to current_peak_render_height
        y_coords_render = np.linspace(0, current_peak_render_height, 120)
        # Calculate normalized y-coordinates for the width profile
        y_coords_normalized_for_width = (y_coords_render / current_peak_render_height if current_peak_render_height > 0 else np.zeros_like(y_coords_render))

        # Use a power-law profile for the width, making tips slimmer
        width_profile_factor = (4 * y_coords_normalized_for_width * (1 - y_coords_normalized_for_width)) ** needle_shape_power

        # --- Plot the GLOW effect first ---
        glow_current_widths_nm = scaled_max_width_nm * glow_width_multiplier * width_profile_factor
        glow_x_left = utils_for_plotly_rewrites.left_x(peak_nm, glow_current_widths_nm)
        glow_x_right = utils_for_plotly_rewrites.right_x(peak_nm, glow_current_widths_nm)


        loop_x = np.concatenate([glow_x_left, glow_x_right[::-1]])
        loop_y = np.concatenate([y_coords_render, y_coords_render[::-1]])

        color_str = f"rgb({color_rgb[0] * 255},{color_rgb[1] *255},{color_rgb[2] *255})" 

        # 3. Add the fill trace to the figure
        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=color_str,
            opacity=glow_alpha,
            line=dict(width=0),      # Replaces edgecolor='none' and linewidth=0
            mode='none',             # Hides the bounding lines completely
            showlegend=False,
            hoverinfo='skip'         # Prevents background element hover tooltips
        ))
                # --- Plot the main NEEDLE on top of the glow ---
        current_widths_nm = utils_for_plotly_rewrites.dynamic_prominence(scaled_max_width_nm, width_profile_factor)
        x_left = utils_for_plotly_rewrites.left_x(peak_nm, current_widths_nm)
        x_right = utils_for_plotly_rewrites.right_x(peak_nm, current_widths_nm)
        loop_x = np.concatenate([x_left, x_right[::-1]])
        loop_y = np.concatenate([y_coords_render, y_coords_render[::-1]])



        # 3. Add to the figure (execute this AFTER the zorder=1 trace)
        trad_fig.add_trace(go.Scatter(
            x=loop_x,
            y=loop_y,
            fill='toself',
            fillcolor=color_str,
            #opacity=glow_alpha,
            line=dict(width=0),      # Replaces edgecolor='none' and linewidth=0
            mode='none',             # Smooth fill with no outline paths
            showlegend=False,
            hoverinfo='skip'         # Keeps hover data focused on foreground lines
        ))

        # Add text label for the peak wavelength with rotation and stroke for readability
        if show_peak_labels and peak_norm_int >= label_min_norm_int:
            y_for_label = label_ys[j] if j < len(label_ys) else peak_label_y_position
            
            trad_fig.add_annotation(
                x=peak_nm,
                y=y_for_label,
                text=f"{peak_nm:.2f}",
                font=dict(
                    family='DejaVu Sans, sans-serif',
                    weight=100,
                    color="white",
                    size=9  # Matplotlib size 8 translates roughly to 10-11px in Plotly
                ),
                showarrow=False,
                textangle=-60,      # Plotly rotates clockwise; -60 equivalent to +60 counter-clockwise
                xanchor="center",     
                yanchor="middle",   

                xshift=5,
                yshift=25,
                
                # Replicates path_effects=[pe.withStroke(linewidth=1.5, foreground="black")]
                # Uses CSS text-shadow to create a crisp 1.5px black outline
                captureevents=False # Ensures annotation doesn't block plot clicks
            )
            
            # Apply the text stroke using Plotly's HTML template support
            # This edits the last added annotation directly to inject the CSS shadow style
            trad_fig.layout.annotations[-1].font.color = "white"
            trad_fig.layout.annotations[-1].text = (
                f"<span style='text-shadow: "
                f"-1.5px -1.5px 0 #000, 1.5px -1.5px 0 #000, "
                f"-1.5px 1.5px 0 #000, 1.5px 1.5px 0 #000;'>"
                f"{peak_nm:.2f}</span>"
            )


    #plt.tight_layout(rect=[0, 0, 1, 0.92])

    trad_fig.update_layout(
        # Replicates plt.tight_layout() by auto-fitting margins around labels
        xaxis=dict(automargin=True),
        yaxis=dict(automargin=False),
        
        # Replicates rect=[0, 0, 1, 0.92]
        # Restricts the plot plotting area by clearing extra padding space
        margin=dict(
            l=10,  # Left margin in pixels
            r=10,  # Right margin in pixels
            b=10,  # Bottom margin in pixels
            t=40   # Top margin in pixels (leaves a ~8% gap at top for titles/labels)
        )
    )
    return trad_fig

In [18]:
def trad_spec(
    data_df, 
    #mode,
    scale_mode='raw',
    show_peak_labels=True,
    nm_col=None,
    int_col=None,
    detect_columns=False,
    title = None,
    random_title = None,
    show_grid = False,
    show_label_colour = None,
    scale_by_int = None,
    fig_size=utils_for_plotly_rewrites.FIG_SIZE, 
    dpi=utils_for_plotly_rewrites.DPI,
):
    df_plot_data, should_exit_early, has_any_nist = prep_with_nist(data_df = data_df, nm_col=nm_col, int_col=int_col, detect_columns=detect_columns)

    fig = go.Figure()
    fig = plot_trad(
        df_plot_data = df_plot_data, 
        nm_col = wl_col, 
        has_any_nist=has_any_nist,
        scale_mode=scale_mode,
        show_peak_labels=show_peak_labels,
        detect_columns=detect_columns,
        title=title,
        random_title=random_title,
    )
    fig.update_layout(trad_spec_labels(has_any_nist=has_any_nist, scale_mode=scale_mode))

    config = {
        'toImageButtonOptions': {
            'format': 'png',
            'scale': 10.0,
            'filename': f"{utils_for_plotly_rewrites.generate_random_title()}" 
        }
    }



    fig.show(config=config)

In [32]:
trad_spec(NIST_DF)

NIST Destriptors Detected. Preparing NIST Rendering.
(prep_with_nist) ymax =  490.0
DEBUG: has nist plot trad True

overflow: 0.19999999999999996
o.g. peak label y pos'n: 0.78

og max needle y: 0.77 
max label y: 0.98 
init peak label y: 0.78 
	 label ys [0.78, 0.78, 0.78, 0.78, 0.86, 0.9400000000000001, 0.8466666666666667, 0.9133333333333333, 0.98, 0.78, 0.9, 0.86, 0.98, 0.8200000000000001, 0.78, 0.94, 0.8383333333333334, 0.905, 0.78, 0.8466666666666667, 0.955, 0.7883333333333333, 0.805, 0.9299999999999999, 0.855, 0.9383333333333334, 0.8633333333333333, 0.9633333333333334, 0.8133333333333334, 0.8216666666666667, 0.8883333333333333, 0.8966666666666667, 0.8300000000000001, 0.7966666666666666, 0.9716666666666667, 0.9133333333333333, 0.98, 0.9466666666666667, 0.8716666666666667, 0.88, 0.9216666666666666, 0.8300000000000001, 0.88, 0.9299999999999999, 0.98, 0.78, 0.86, 0.78, 0.8085714285714286, 0.8371428571428572, 0.9228571428571428, 0.98, 0.8657142857142857, 0.8942857142857142, 0.951428571